In [1]:
!pip -q install biopython

import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices("GPU"))

import os
import sys
import math
import random
import numpy as np
import json
import math
import argparse
import csv
import hashlib

from itertools import product
from Bio import SeqIO
from pathlib import Path
from typing import Optional, List
from dataclasses import dataclass, asdict
from datetime import datetime

from tensorflow.keras.models import model_from_json
from tensorflow.keras.optimizers import RMSprop

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 43.3 MB/s eta 0:00:00
TensorFlow: 2.20.0
GPU disponible: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


#DEEPDNA

In [2]:
class RangeEncoder:
    def __init__(self):
        self.low = 0
        self.high = 0xFFFFFFFF
        self.pending_bits = 0  # CRITICAL FIX: Tracks underflow bits
        self.bits = []

    def encode_symbol(self, symbol, freqs):
        total = sum(freqs.values())
        cum_low = 0
        keys_ordre = ['A', 'C', 'G', 'T']

        for k in keys_ordre:
            if k == symbol:
                break
            cum_low += freqs[k]
        cum_high = cum_low + freqs[symbol]

        Range = self.high - self.low + 1
        self.high = self.low + (Range * cum_high // total) - 1
        self.low = self.low + (Range * cum_low // total)

        while True:
            if (self.low ^ self.high) & 0x80000000 == 0:
                # MSBs match. Write bit, then write pending opposite bits.
                bit = self.low >> 31
                self.bits.append(bit)
                while self.pending_bits > 0:
                    self.bits.append(bit ^ 1)
                    self.pending_bits -= 1

                self.low = (self.low << 1) & 0xFFFFFFFF
                self.high = ((self.high << 1) | 1) & 0xFFFFFFFF

            elif (self.low & 0x40000000) and not (self.high & 0x40000000):
                # Straddle/Underflow! Zoom in but save the pending bit
                self.pending_bits += 1
                self.low = (self.low << 1) & 0x7FFFFFFF
                self.high = ((self.high << 1) | 0x80000001) & 0xFFFFFFFF
            else:
                break

    def finish(self):
        # Safely flush the remaining state to the file
        self.pending_bits += 1
        if self.low < 0x40000000:
            self.bits.append(0)
            for _ in range(self.pending_bits):
                self.bits.append(1)
        else:
            self.bits.append(1)
            for _ in range(self.pending_bits):
                self.bits.append(0)
        return self.bits


class RangeDecoder:
    def __init__(self, bit_iterator):
        self.low = 0
        self.high = 0xFFFFFFFF
        self.value = 0
        self.bit_iterator = bit_iterator

        for _ in range(32):
            try:
                bit = next(self.bit_iterator)
            except StopIteration:
                bit = 0
            self.value = ((self.value << 1) | bit) & 0xFFFFFFFF

    def decode_symbol(self, freqs):
        total = sum(freqs.values())
        Range = self.high - self.low + 1
        cum_target = ((self.value - self.low + 1) * total - 1) // Range

        cum = 0
        keys_ordre = ['A', 'C', 'G', 'T']
        chosen_symbol = 'A'
        cum_low = 0
        cum_high = freqs['A']

        for k in keys_ordre:
            if cum <= cum_target < cum + freqs[k]:
                chosen_symbol = k
                cum_low = cum
                cum_high = cum + freqs[k]
                break
            cum += freqs[k]

        self.high = self.low + (Range * cum_high // total) - 1
        self.low = self.low + (Range * cum_low // total)

        while True:
            if (self.low ^ self.high) & 0x80000000 == 0:
                self.low = (self.low << 1) & 0xFFFFFFFF
                self.high = ((self.high << 1) | 1) & 0xFFFFFFFF
                try: bit = next(self.bit_iterator)
                except StopIteration: bit = 0
                self.value = ((self.value << 1) | bit) & 0xFFFFFFFF

            elif (self.low & 0x40000000) and not (self.high & 0x40000000):
                self.low = (self.low << 1) & 0x7FFFFFFF
                self.high = ((self.high << 1) | 0x80000001) & 0xFFFFFFFF
                try: bit = next(self.bit_iterator)
                except StopIteration: bit = 0

                # CRITICAL FIX: Preserve the MSB, shift only the 2nd MSB and below
                self.value = (self.value & 0x80000000) | ((self.value << 1) & 0x7FFFFFFF) | bit
            else:
                break

        return chosen_symbol

In [3]:
CHARS = "ACGT"
CHAR_INDICES = {c: i for i, c in enumerate(CHARS)}
INDICES_CHAR = {i: c for i, c in enumerate(CHARS)}
MAX_LEN = 64
input_dim = len(CHARS)

# Factor d'escala per transformar floats de probabilitat [0.0, 1.0] a freqüències senceres de Nayuki
SCALE_FACTOR = 1000000  # 1 Milions per garantir Lossless complet


np.random.seed(1337)
tf.random.set_seed(1337)
random.seed(1337)

def load_model_from_json(json_path="./model/model.json", weights_path="./model/model_latest.weights.h5"):
    if not os.path.exists(json_path) or not os.path.exists(weights_path):
        print(f"Error: No es troben els fitxers del model a:\n\t{json_path}\n\t{weights_path}")
        sys.exit(1)

    with open(json_path, "r") as json_file:
        loaded_model_json = json_file.read()

    model = model_from_json(loaded_model_json)
    model.load_weights(weights_path)

    optimizer = RMSprop(learning_rate=0.001)
    model.compile(loss="categorical_crossentropy", optimizer=optimizer)

    print(f"Model carregat des de {json_path} i {weights_path}")
    return model

def prepare_input_window(window_string):
    """Converteix una finestra de text en la teva matriu 3D One-Hot per a inferència."""
    x = np.zeros((1, MAX_LEN, len(CHARS)), dtype=bool)
    for t, char in enumerate(window_string):
        x[0, t, CHAR_INDICES[char]] = 1
    return x


## PREPROCESS I POSTPROCESS HEADERS / METADATA FASTA


In [4]:

def preprocess_fasta(data_path):
    """
    Llegeix el fitxer FASTA guardant capçaleres i caràcters modificats
    per garantir una descompressió idèntica.
    """
    # Guardem la capçalera original (p.ex: >chrM)
    raw_seq = ""
    header = ""
    with open(data_path, "r", encoding="utf-8", errors="ignore") as f:
        header_line = f.readline()
        if not header_line.startswith(">"):
            print(f"Error: El fitxer {data_path} no sembla ser un FASTA vàlid (no comença per >).")
            sys.exit(1)

        header = header_line

        for line in f:
            if line.startswith(">"):
                print(f"Error: El fitxer {data_path} sembla ser un multi-FASTA, cosa que no està suportada.")
                sys.exit(1)

            raw_seq += line.strip()

    metadata_log = []
    cleaned_sequence = []

    # Processat caràcter per caràcter d'acord amb els chars admessos (chars="ACGT")
    for idx, char in enumerate(raw_seq):
        if char in CHARS:
            cleaned_sequence.append(char)
        else:
            # Guardem la posició i el caràcter original (minúscules, N, wildcards...)
            metadata_log.append((idx, char))

    cleaned_text = "".join(cleaned_sequence)

    if len(cleaned_text) <= MAX_LEN:
        raise ValueError(f"La seqüència és massa curta per al context del model ({len(cleaned_text)} bases).")

    return header, cleaned_text, metadata_log

def postprocess_fasta(cleaned_text, metadata_log):
    seq_list = list(cleaned_text)
    # Substitució directa lossless per índex
    for idx, original_char in metadata_log:
        if idx < len(seq_list):
            seq_list.insert(idx, original_char)
    original_sequence = "".join(seq_list)
    return original_sequence


In [5]:

def compression_arithmetic_network(cleaned_text, output_file_path, model):
    total_bases = len(cleaned_text)

    print("[*] Encodificant el flux de bits...")
    encoder = RangeEncoder()

    for i in range(total_bases):
        if i % 5000 == 0 and i > 0:
          print(f"    -> Codificades {i}/{total_bases} bases...")
        # 1. Build the window exactly like the decompressor does
        if i < MAX_LEN:
            padding_needed = MAX_LEN - i
            window = ("A" * padding_needed) + cleaned_text[:i]
        else:
            window = cleaned_text[i - MAX_LEN : i]

        # 2. Get the probability sequentially
        x_input = prepare_input_window(window)
        prob_dist = model(x_input, training=False)[0].numpy()
        char = cleaned_text[i]

        # 3. Scale to integer frequencies
        frequencies_dinamiques = {
            'A': max(1, round(prob_dist[0] * SCALE_FACTOR)),
            'C': max(1, round(prob_dist[1] * SCALE_FACTOR)),
            'G': max(1, round(prob_dist[2] * SCALE_FACTOR)),
            'T': max(1, round(prob_dist[3] * SCALE_FACTOR))
        }

        encoder.encode_symbol(char, frequencies_dinamiques)
    compressed_bits = encoder.finish()

    # Guardem a disc empaquetant els bits en bytes de veritat
    print("[*] Guardant fitxer binari resultant (.ddna)...")
    raw_bytes = bytearray()
    for i in range(0, len(compressed_bits), 8):
        byte_chunk = compressed_bits[i : i + 8]
        if len(byte_chunk) < 8:
            byte_chunk = byte_chunk + [0] * (8 - len(byte_chunk))
        byte_str = "".join(str(b) for b in byte_chunk)
        raw_bytes.append(int(byte_str, 2))

    return raw_bytes

def run_compression(input_path, output_dir, model):
  # preprocess
  header, cleaned_text, metadata_log = preprocess_fasta(input_path)

  os.makedirs(output_dir, exist_ok=True)
  base_name = os.path.basename(input_path)
  output_file_path = os.path.join(output_dir, f"{base_name}.ddna")

  raw_bytes = compression_arithmetic_network(cleaned_text, output_file_path, model)

  with open(output_file_path, "wb") as out_f:
    metadata = {
        "header": header,
        "metadata_log": metadata_log,
        "length": len(cleaned_text)
    }
    meta_bytes = json.dumps(metadata).encode('utf-8')
    out_f.write(len(meta_bytes).to_bytes(4, byteorder='big'))
    out_f.write(meta_bytes)
    out_f.write(raw_bytes)
  print("[*] Compressió finalitzada a", output_file_path)
  return output_file_path

In [6]:
def decompression_arithmetic_network(file_bytes, expected_length, model):
  # Recuperem la llista de bits del fitxer
  bit_list = []
  for byte in file_bytes:
    bits_string = f"{byte:08b}"
    for bit in bits_string:
      bit_list.append(int(bit))

  bit_iterator = iter(bit_list)
  cleaned_text_list = []

  # Inicialitzem el descodificador de rangs continu
  decoder = RangeDecoder(bit_iterator)

  print(f"[*] Reconstruint seqüència base per base amb la GPU ({expected_length} bases)...")

  # Bucle controlat mil·limètricament per la longitud real
  for i in range(expected_length):
    if i % 2000 == 0 and i > 0:
      print(f"    -> Descodificades {i}/{expected_length} bases...")

    if i < MAX_LEN:
      padding_needed = MAX_LEN - i
      window = ("A" * padding_needed) + "".join(cleaned_text_list[:i])
    else:
      window = "".join(cleaned_text_list[i - MAX_LEN : i])

    x_input = prepare_input_window(window)

    # Use the exact same execution method
    prob_dist = model(x_input, training=False)[0].numpy()

    frequencies = {
        'A': max(1, round(prob_dist[0] * SCALE_FACTOR)),
        'C': max(1, round(prob_dist[1] * SCALE_FACTOR)),
        'G': max(1, round(prob_dist[2] * SCALE_FACTOR)),
        'T': max(1, round(prob_dist[3] * SCALE_FACTOR))
    }

    # Descodifiquem el caràcter segons el nostre motor adaptatiu
    caracter = decoder.decode_symbol(frequencies)
    cleaned_text_list.append(caracter)

  cleaned_text = "".join(cleaned_text_list)
  return cleaned_text

def run_decompression(compressed_path, decompressed_dir, model, chars_line_fasta=70):
  os.makedirs(decompressed_dir, exist_ok=True)
  base_name = os.path.basename(compressed_path).replace('.ddna', '')
  output_file_path = os.path.join(decompressed_dir, base_name)

  print(f"[*] Descomprimint i descodificant el fitxer binari: {compressed_path}")

  with open(compressed_path, "rb") as in_f:
    meta_len = int.from_bytes(in_f.read(4), byteorder='big')
    metadata = json.loads(in_f.read(meta_len).decode('utf-8'))
    print(metadata)
    header = metadata["header"]
    metadata_log = metadata["metadata_log"]
    expected_length = metadata["length"]

    file_bytes = in_f.read()

  cleaned_text = decompression_arithmetic_network(file_bytes, expected_length, model)

  original_sequence = postprocess_fasta(cleaned_text, metadata_log)

  with open(output_file_path, "w") as out_f:
    out_f.write(f"{header}")
    for i in range(0, len(original_sequence), chars_line_fasta):
      out_f.write(original_sequence[i : i + chars_line_fasta] + "\n")

  return output_file_path



# ENTITIES AND UTILS

In [7]:
@dataclass
class Dataset:
    dataset_id: str # num o codi curt
    label: str # categoria
    description: str
    fasta_path: Path
    size_bytes: int
    num_bases_std: int # només A,C,G,T
    num_bases_seq: int # A,C,G,T,N i unknown
    base_counts: dict[str, int] # counts A,C,G,T,N i unknown
    size_header: float # quants caracters te el header
    header_counts: dict[str, int] # counts de caracters del header
    entropy_acgtn_bits_per_base: float | None # Entropia H0 sobre A,C,G,T,N
    entropy_acgt_bits_per_base: float | None # Entropia H0 sobre A,C,G,T ignorant N
    entropy_seq_all_bits_per_base: float | None # Entropia H0 sobre la seqüència completa
    entropy_seq_all_headers_bits_per_base: float | None # Entropia H0 sobre la seqüència completa i els headers
    contains_unknown: bool = False

@dataclass
class Algorithm:
    name: str
    compress_cmd: str
    decompress_cmd: str
    compressed_ext: str

@dataclass
class ResultRow:
    dataset_id: str
    dataset_label: str
    algorithm: str
    original_size_bytes: int
    compressed_size_bytes: int
    compression_ratio: float
    compression_percent: float
    bits_per_base_seq: float # eficiència sobre ADN pur (només seqüència)
    size_header: float
    entropy_acgtn_bpb: float | None # Entropia H0 sobre A,C,G,T,N
    entropy_acgt_bpb: float | None # Entropia H0 sobre A,C,G,T ignorant N
    entropy_seq_bpb: float | None # Entropia H0 sobre la seqüència completa
    entropy_all_bpb: float | None # Entropia H0 sobre la seqüència completa i els headers
    compress_seconds: float
    decompress_seconds: float
    compress_MBps: float
    decompress_MBps: float
    bitwise_ok: bool
    sequence_ok_all: bool
    sequence_ok_std: bool
    compress_exit_code: int
    decompress_exit_code: int
    stderr_log: str
    status: str
    num_bases_seq: int
    num_A: int
    num_C: int
    num_G: int
    num_T: int
    num_N: int = 0
    num_unknown: int = 0

@dataclass
class BenchmarkPaths:
    compressed_dir: Path
    decompressed_dir: Path
    logs_dir: Path
    tmp_dir: Path


def load_datasets(DATA_DIR: Path) -> List[Dataset]:
    """
    Automatically loads all FASTA files within data,
    including subdirectories such as virus, bacteria, mitochondria, chromosome and genome.
    """
    print(f"Loading datasets from {DATA_DIR}...")
    fasta_extensions = {".fasta", ".fa"} # ".fna"}
    items = []

    not_run = "chromosome" # "genome" # "chromosome" # "mitochondria" # "bacteria" # "virus"

    for fasta_path in sorted(DATA_DIR.rglob("*")):
        # if not_run in fasta_path.parts:
        #     continue
        if not fasta_path.is_file():
            continue

        if fasta_path.suffix.lower() not in fasta_extensions:
            continue

        dataset_type = fasta_path.parent.name
        dataset_id = fasta_path.stem
        description = read_fasta_description(fasta_path)

        base_counts = count_fasta_symbols(fasta_path)
        num_bases = sum(base_counts[base] for base in "ACGT")
        num_no_std = sum(base_counts.get("unknown", {}).values()) + base_counts.get("N", 0)

        size_header = count_size_header(fasta_path)

        entropies = compute_dataset_entropies(fasta_path, base_counts)
        entropy_acgtn = entropies["entropy_acgtn"]
        entropy_acgt = entropies["entropy_acgt"]
        entropy_seq_all = entropies["entropy_seq_all"]
        entropy_seq_all_headers = entropies["entropy_seq_headers"]
        header_counts = entropies["header_counts"]

        items.append(Dataset(
            dataset_id=dataset_id,
            label=dataset_type,
            description=description,
            fasta_path=fasta_path,
            size_bytes=file_size(fasta_path),
            contains_unknown=(num_no_std > 0),
            num_bases_std=num_bases,
            num_bases_seq=num_bases + num_no_std,
            base_counts=base_counts,
            size_header=size_header,
            header_counts=header_counts,
            entropy_acgtn_bits_per_base=entropy_acgtn,
            entropy_acgt_bits_per_base=entropy_acgt,
            entropy_seq_all_bits_per_base=entropy_seq_all,
            entropy_seq_all_headers_bits_per_base=entropy_seq_all_headers

        ))

    return items


In [8]:
def count_total_bases(fasta_path: Path) -> int:
    total = 0
    with open(fasta_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.startswith(">"):
                total += sum(1 for ch in line.strip().upper() if ch in "ACGTN")
    return total

def count_N_bases(fasta_path: Path) -> int:
    total = 0
    with open(fasta_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.startswith(">"):
                total += line.upper().count("N")
    return total

def count_size_header(fasta_path: Path) -> int:
    size_header = 0
    with open(fasta_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if line.startswith(">"):
                size_header += len(line.encode("utf-8"))
            else:
                continue # per considerar tambe multi-fasta
    return size_header

def count_fasta_symbols(fasta_path: Path) -> dict[str, int]:
    """
    Compta els símbols A, C, G, T i N de la seqüència FASTA, ignorant capçaleres i salts de línia.
    """
    accepted_bases = set("ACGTN")
    counts = {base: 0 for base in set(accepted_bases | {"unknown"})}
    counts["unknown"] = dict()
    with open(fasta_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if line.startswith(">"):
                continue
            for ch in line.strip().upper():
                if ch in accepted_bases:
                    counts[ch] += 1
                else:
                    counts["unknown"][ch] = counts["unknown"].get(ch, 0) + 1
    return counts

def count_header_symbols(fasta_path: Path) -> dict[str, int]:
    counts = {}

    with open(fasta_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if line.startswith(">"):
                for ch in line.strip():
                    counts[ch] = counts.get(ch, 0) + 1

    return counts

def read_fasta_description(fasta_path: Path) -> str:
    """
    Llegeix la primera capçalera FASTA i retorna la descripció,
    és a dir, tot el que ve després del primer identificador.

    Exemple:
    >U00096.3 Escherichia coli str. K-12 substr. MG1655, complete genome

    retorna:
    Escherichia coli str. K-12 substr. MG1655, complete genome
    """
    with open(fasta_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if line.startswith(">"):
                header = line[1:].strip()
                parts = header.split(maxsplit=1)

                if len(parts) == 2:
                    return parts[1]
                else:
                    return parts[0]

    return fasta_path.stem.replace("_", " ")

def delete_fasta_header(fasta_path: Path, output_path: Path) -> None:
    """
    Crea una nova versió del fitxer FASTA sense les línies de capçalera (les que comencen amb '>').
    """
    with open(fasta_path, "r", encoding="utf-8", errors="ignore") as infile, \
         open(output_path, "w", encoding="utf-8") as outfile:
        for line in infile:
            if not line.startswith(">"):
                outfile.write(line)



In [9]:

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """
    Calcula empremta digital SHA-256 d'un fitxer llegint-lo en trossos per evitar carregar-lo tot a la memòria.
    """
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def file_size(path: Path) -> int:
    return path.stat().st_size

def mbps(num_bytes: int, seconds: float) -> float:
    if seconds <= 0:
        return 0.0
    return (num_bytes / (1024 * 1024)) / seconds

def shannon_entropy_from_counts(counts: dict[str, int]) -> Optional[float]:
    """
    Entropia de Shannon d'ordre 0, en bits per símbol: H = -sum p_i log2(p_i).
    Retorna None si no hi ha símbols.
    """
    total = sum(counts.values())
    if total == 0:
        return None

    entropy = 0.0
    for count in counts.values():
        if count == 0:
            continue
        p = count / total
        entropy -= p * math.log2(p)
    return entropy

def compute_dataset_entropies(fasta_path: Path, base_counts: dict[str, int]) -> dict[str, Optional[float]]:

    # (a) ACGT
    acgt_counts = {k: base_counts.get(k, 0) for k in "ACGT"}

    # (b) ACGTN
    acgtn_counts = {k: base_counts.get(k, 0) for k in "ACGTN"}

    # (c) tots els símbols de seqüència (inclou unknown si el tens)
    seq_all_counts = acgtn_counts.copy()
    for k, v in base_counts.get("unknown", {}).items():
        seq_all_counts[k] = v

    # (d) seq + headers
    header = count_header_symbols(fasta_path)
    for k, v in header.items():
        seq_all_counts[k] = seq_all_counts.get(k, 0) + v
    header_counts = seq_all_counts.copy()

    return {
        "entropy_acgt": shannon_entropy_from_counts(acgt_counts),
        "entropy_acgtn": shannon_entropy_from_counts(acgtn_counts),
        "entropy_seq_all": shannon_entropy_from_counts(seq_all_counts),
        "entropy_seq_headers": shannon_entropy_from_counts(header_counts),
        "header_counts": header_counts
    }

def extract_sequence(path: Path) -> str:
    seq = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if not line.startswith(">"):
                seq.append(line.strip().upper())
    return "".join(seq)

def remove_N(seq: str) -> str:
    return seq.replace("N", "")

def remove_unknown(seq: str) -> str:
    return "".join(base for base in seq if base in "ACGT")

def write_csv(rows: List[ResultRow], out_csv: Path):
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(asdict(rows[0]).keys()))
        w.writeheader()
        for r in rows:
            w.writerow(asdict(r))

# BENCHMARK

In [10]:
#!/usr/bin/env python3

import shlex
import shutil
import subprocess
import time
from pathlib import Path

def compress_command(input_path: Path, output_dir: Path, model) -> str:
    start = time.perf_counter()
    file = run_compression(input_path, output_dir, model)
    total_time = time.perf_counter() - start
    if file:
      return 0, total_time
    else:
      return 1, total_time

def decompress_command(compressed_path: Path, decompressed_dir: Path, model) -> str:
    start = time.perf_counter()
    file = run_decompression(compressed_path, decompressed_dir, model)
    total_time = time.perf_counter() - start
    if file:
      return 0, total_time
    else:
      return 1, total_time



def validate_decompression(input_path: Path, decompressed_path: Path, num_no_std: int) -> tuple[bool, bool, bool]:
    """
    Retorna:
    - bitwise_ok: el fitxer és idèntic byte a byte
    - sequence_ok_all: la seqüència FASTA és igual
    - sequence_ok_std: la seqüència és igual ignorant tots els no ACGT
    """
    if not decompressed_path.exists():
        print("Output file does not exist")
        return False, False, False

    bitwise_ok = sha256_file(input_path) == sha256_file(decompressed_path)

    try:
        seq_in = extract_sequence(input_path)
        seq_out = extract_sequence(decompressed_path)

        sequence_ok_all = seq_in == seq_out
        sequence_ok_std = False

        if not sequence_ok_all and num_no_std > 0:
            sequence_ok_std = remove_unknown(seq_in) == remove_unknown(seq_out)
        elif sequence_ok_all:
            sequence_ok_std = True

        return bitwise_ok, sequence_ok_all, sequence_ok_std

    except Exception:
        return bitwise_ok, False, False

def benchmark_one(ds: Dataset, algo: Algorithm, paths: BenchmarkPaths, model) -> ResultRow:
    input_path = ds.fasta_path
    original_size = file_size(input_path)

    num_bases_seq = ds.num_bases_seq
    num_A = ds.base_counts.get("A", 0)
    num_C = ds.base_counts.get("C", 0)
    num_G = ds.base_counts.get("G", 0)
    num_T = ds.base_counts.get("T", 0)
    num_N = ds.base_counts.get("N", 0)
    num_unknown = sum(ds.base_counts.get("unknown", {}).values())

    # Entropia teòrica d'ordre 0 de la seqüència.
    # H0(A,C,G,T,N): útil si vols considerar N com un símbol més.
    # H0(A,C,G,T): útil si vols analitzar només les bases conegudes.
    size_header = ds.size_header
    entropy_acgtn = ds.entropy_acgtn_bits_per_base
    entropy_acgt = ds.entropy_acgt_bits_per_base
    entropy_seq = ds.entropy_seq_all_bits_per_base
    entropy_all = ds.entropy_seq_all_headers_bits_per_base

    compressed_path = paths.compressed_dir /ds.dataset_id / algo.name / f"{input_path.name}{algo.compressed_ext}"
    compressed_dir = paths.compressed_dir /ds.dataset_id / algo.name
    decompressed_dir = paths.decompressed_dir / ds.dataset_id / algo.name
    decompressed_path = paths.decompressed_dir / ds.dataset_id / algo.name / input_path.name
    stderr_path = paths.logs_dir / f"{ds.dataset_id}.{algo.name}.stderr.txt"

    for p in [compressed_path, decompressed_path, stderr_path]:
        p.parent.mkdir(parents=True, exist_ok=True)

    if compressed_path.exists():
        compressed_path.unlink()

    if decompressed_path.exists():
        decompressed_path.unlink()

    if stderr_path.exists():
        stderr_path.unlink()

    c_code, compress_seconds = compress_command(
        input_path,
        compressed_dir,
        model
    )

    compressed_size = file_size(compressed_path) if compressed_path.exists() else -1

    d_code = -1
    decompress_seconds = 0.0

    if compressed_path.exists() and c_code == 0:
        d_code, decompress_seconds = decompress_command(
            compressed_path,
            decompressed_dir,
            model
        )

    bitwise_ok, sequence_ok_all, sequence_ok_std = validate_decompression(
        input_path=input_path,
        decompressed_path=decompressed_path,
        num_no_std=num_N + num_unknown,
    )

    ratio = (original_size / compressed_size) if compressed_size > 0 else None
    percent = (100.0 * (1.0 - compressed_size / original_size)) if compressed_size > 0 else None

    # Incloent headers (fitxer complet)
    bits_per_base_total = (
        (compressed_size * 8.0) / original_size
    ) if compressed_size > 0 and original_size > 0 else None

    # Només seqüència
    bits_per_base_seq = (
        (compressed_size * 8.0) / num_bases_seq
    ) if compressed_size > 0 and num_bases_seq > 0 else None

    status = "ok" if (c_code == 0 and d_code == 0 and sequence_ok_all) else (
        "ok_std" if (c_code == 0 and d_code == 0 and sequence_ok_std) else (
            "compress_error" if c_code != 0 else (
                "decompress_error" if d_code != 0 else (
                    "mismatch"
                )
            )
        )
    )

    return ResultRow(
        dataset_id=ds.dataset_id,
        dataset_label=ds.label,
        algorithm=algo.name,
        original_size_bytes=original_size,
        compressed_size_bytes=compressed_size,
        compression_ratio=ratio,
        compression_percent=percent,
        bits_per_base_seq=bits_per_base_seq,
        size_header=size_header,
        entropy_acgtn_bpb=entropy_acgtn,
        entropy_acgt_bpb=entropy_acgt,
        entropy_seq_bpb=entropy_seq,
        entropy_all_bpb=entropy_all,
        compress_seconds=compress_seconds,
        decompress_seconds=decompress_seconds,
        compress_MBps=mbps(original_size, compress_seconds),
        decompress_MBps=mbps(original_size, decompress_seconds),
        bitwise_ok=bitwise_ok,
        sequence_ok_all=sequence_ok_all,
        sequence_ok_std=sequence_ok_std,
        compress_exit_code=c_code,
        decompress_exit_code=d_code,
        stderr_log=str(stderr_path),
        status=status,
        num_bases_seq=num_bases_seq,
        num_A=num_A,
        num_C=num_C,
        num_G=num_G,
        num_T=num_T,
        num_N=num_N,
        num_unknown=num_unknown,
    )

# COMPARAR DOS FITXERS

In [11]:
import hashlib
from pathlib import Path
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """
    Calcula empremta digital SHA-256 d'un fitxer llegint-lo en trossos per evitar carregar-lo tot a la memòria.
    """
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

In [12]:
import sys
from Bio import SeqIO

def comparar_fasta_lossless(ruta1, ruta2):
    # 1. Carreguem els registres bioinformàtics dels dos fitxers
    records1 = list(SeqIO.parse(ruta1, "fasta"))
    records2 = list(SeqIO.parse(ruta2, "fasta"))
    print(records1[0])
    print(records2[0])

    # Validació inicial: Comprovem si tenen el mateix nombre de seqüències
    if len(records1) != len(records2):
        print(f"❌ ERROR: Els fitxers tenen un nombre diferent de seqüències!")
        print(f"  - {ruta1}: {len(records1)} seqüències")
        print(f"  - {ruta2}: {len(records2)} seqüències")
        return False

    print(f"[*] Començant comparació biològica entre:\n  F1: {ruta1}\n  F2: {ruta2}\n" + "-"*50)

    tots_identics = True

    # 2. Iterem parella per parella de seqüències
    for i, (rec1, rec2) in enumerate(zip(records1, records2)):
        print(f"\n[+] Analitzant seqüència #{i+1}:")

        # --- PAS A: COMPARACIÓ DE HEADERS ---
        # Reconstruïm la capçalera completa (>ID Descripció) per comparar-les
        header1 = f">{rec1.id} {rec1.description}".strip()
        header2 = f">{rec2.id} {rec2.description}".strip()

        if header1 == header2:
            print(f"  ✅ Headers idèntics: {header1}")
        else:
            print(f"  ❌ DIFERÈNCIA ALS HEADERS:")
            print(f"    - Original:     {header1}")
            print(f"    - Descomprimit: {header2}")
            tots_identics = False

        # --- PAS B: COMPARACIÓ DE LA SEQÜÈNCIA SENCERA ---
        # Convertim les seqüències a string de majúscules per evitar falsos negatius de format
        seq1_str = str(rec1.seq)
        seq2_str = str(rec2.seq)

        if seq1_str == seq2_str:
            print(f"  ✅ Seqüències idèntiques ({len(seq1_str)} bases).")
        else:
            print(f"  ❌ DIFERÈNCIA A LA SEQÜÈNCIA DE BASES!")
            tots_identics = False

            # Si vols veure on comença a fallar, busquem el primer caràcter diferent:
            mida_minima = min(len(seq1_str), len(seq2_str))
            for idx in range(mida_minima):
                if seq1_str[idx] != seq2_str[idx]:
                    print(f"    - Primer error trobat a la base posició: {idx}")
                    print(f"    - Text original:     ... {seq1_str[max(0, idx-10):idx+10]} ...")
                    print(f"    - Text descomprimit: ... {seq2_str[max(0, idx-10):idx+10]} ...")
                    break
            # print(f"    - Text original:     ... {seq1_str[0:50]} ...")
            # print(f"    - Text descomprimit: ... {seq2_str[0:50]} ...")
            break
            if len(seq1_str) != len(seq2_str):
                print(f"    - Alerta: Les mides no quadren! Original: {len(seq1_str)}b vs Descomprimit: {len(seq2_str)}b")

    print("\n" + "="*50)
    if tots_identics:
        print("🎉ÈXIT TOTAL: Els dos fitxers FASTA són 100% idèntics (Headers i Seqüències).")
        return True
    else:
        print("⚠️ ALERTA: S'han detectat diferències entre els fitxers FASTA.")
        return False

# Executa la funció amb les teves rutes:
#comparar_fasta_lossless(input_data, decompressed_path)

In [13]:
def file_size(path):
    return path.stat().st_size

In [14]:
def run_compression_static(fasta_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    base_name = os.path.basename(fasta_path)
    output_file_path = os.path.join(output_dir, f"{base_name}_static.ddna")

    header, cleaned_text, metadata_log = preprocess_fasta(fasta_path)
    total_bases = len(cleaned_text)

    # Calcular freqüències estàtiques (Model d'ordre 0)
    counts = {c: cleaned_text.count(c) for c in CHARS}
    # Assegurem que no hi hagi zeros per evitar problemes (Nayuki)
    frequencies = {c: max(1, counts[c]) for c in CHARS}

    print(f"[*] Comprimint amb model estàtic (freqüències: {frequencies})")

    encoder = RangeEncoder()
    for char in cleaned_text:
        encoder.encode_symbol(char, frequencies)

    compressed_bits = encoder.finish()

    # Convertir a bytes
    raw_bytes = bytearray()
    for i in range(0, len(compressed_bits), 8):
        byte_chunk = compressed_bits[i : i + 8]
        if len(byte_chunk) < 8: byte_chunk = byte_chunk + [0] * (8 - len(byte_chunk))
        raw_bytes.append(int("".join(str(b) for b in byte_chunk), 2))

    # Guardar metadades (incloent les freqüències per al decoder)
    with open(output_file_path, "wb") as out_f:
        metadata = {
            "header": header,
            "metadata_log": metadata_log,
            "length": total_bases,
            "frequencies": frequencies
        }
        meta_bytes = json.dumps(metadata).encode('utf-8')
        out_f.write(len(meta_bytes).to_bytes(4, byteorder='big'))
        out_f.write(meta_bytes)
        out_f.write(raw_bytes)

    return output_file_path

def run_decompression_static(ddna_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    base_name = os.path.basename(ddna_path).replace('.ddna', '')
    output_file_path = os.path.join(output_dir, base_name)

    with open(ddna_path, "rb") as in_f:
        meta_len = int.from_bytes(in_f.read(4), byteorder='big')
        metadata = json.loads(in_f.read(meta_len).decode('utf-8'))
        header = metadata["header"]
        metadata_log = metadata["metadata_log"]
        expected_length = metadata["length"]
        frequencies = metadata["frequencies"] # Recuperem les freqüències fixes
        file_bytes = in_f.read()

    bit_list = [int(bit) for byte in file_bytes for bit in f"{byte:08b}"]
    decoder = RangeDecoder(iter(bit_list))

    cleaned_text_list = []
    for _ in range(expected_length):
        # Sempre utilitzem les mateixes freqüències estàtiques
        caracter = decoder.decode_symbol(frequencies)
        cleaned_text_list.append(caracter)

    cleaned_text = "".join(cleaned_text_list)
    original_sequence = postprocess_fasta(cleaned_text, metadata_log)

    with open(output_file_path, "w") as out_f:
        out_f.write(f"{header}\n")
        for i in range(0, len(original_sequence), 70):
            out_f.write(original_sequence[i : i + 70] + "\n")

    return output_file_path



# MAIN

In [15]:
model_path = Path("/content/drive/MyDrive/TFG/model")
DATA_DIR = Path("/content/drive/MyDrive/TFG/prova")
output_dir = Path("/content/drive/MyDrive/TFG/data/compressed")
decompressed_dir = Path("/content/drive/MyDrive/TFG/data/decompressed")
default_json = Path("/content/drive/MyDrive/TFG/model/model.json")
default_weights = Path("/content/drive/MyDrive/TFG/model/model_latest.weights.h5")
deepdna_model = load_model_from_json(json_path=default_json, weights_path=default_weights)

algos = [
    Algorithm(
        name="DeepDNA",
        compress_cmd="python3 /home/hpisa/DeepDNA/deepdna_pipeline.py -c -i {input} -o {output} -m /home/hpisa/DeepDNA/model_ACGT",
        decompress_cmd="python3 /home/hpisa/DeepDNA/deepdna_pipeline.py -d -i {input} -o {output} -m /home/hpisa/DeepDNA/model_ACGT",
        compressed_ext=".ddna"
    )
]


Model carregat des de /content/drive/MyDrive/TFG/model/model.json i /content/drive/MyDrive/TFG/model/model_latest.weights.h5


In [16]:
RESULTS_DIR = Path("/content/drive/MyDrive/TFG/results")

TIMESTAMP = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
RUN_DIR = RESULTS_DIR / TIMESTAMP

paths = BenchmarkPaths(
    compressed_dir=RUN_DIR / "compressed",
    decompressed_dir=RUN_DIR / "decompressed",
    logs_dir=RUN_DIR / "logs",
    tmp_dir=RUN_DIR / "tmp",
)

for d in [
    paths.compressed_dir,
    paths.decompressed_dir,
    paths.logs_dir,
    paths.tmp_dir,
]:
    d.mkdir(parents=True, exist_ok=True)

algos = [
    Algorithm(
        name="DeepDNA",
        compress_cmd="python3 /home/hpisa/DeepDNA/deepdna_pipeline.py -c -i {input} -o {output} -m /home/hpisa/DeepDNA/model_ACGT",
        decompress_cmd="python3 /home/hpisa/DeepDNA/deepdna_pipeline.py -d -i {input} -o {output} -m /home/hpisa/DeepDNA/model_ACGT",
        compressed_ext=".ddna"
    )
]

datasets = load_datasets(DATA_DIR)

if not datasets:
    print(f"No FASTA files found in {DATA_DIR}")

if not algos:
    print("No algorithms found in PATH")


print(f"Using {len(datasets)} datasets, {len(algos)} algorithms")

rows = []
for x, ds in enumerate(datasets):
    for y, algo in enumerate(algos):
        print(f"[RUN {x+1}.{y+1}] {ds.dataset_id} / {algo.name}")
        try:
            rows.append(benchmark_one(ds, algo, paths, deepdna_model))
        except Exception as e:
            print(f"[ERROR] {ds.dataset_id} / {algo.name}: {e}")

filename = f"comparative_results_{TIMESTAMP}.csv"

out_csv = RUN_DIR / filename
write_csv(rows, out_csv)
print(f"Results written to {out_csv}")

Loading datasets from /content/drive/MyDrive/TFG/prova...
Using 1 datasets, 1 algorithms
[RUN 1.1] mitochondria / DeepDNA
[*] Encodificant el flux de bits...


KeyboardInterrupt: 